1. ติดตั้งไลบรารีที่จำเป็น

In [ ]:
pip install torch transformers scikit-learn pandas numpy datasets

Base Models

In [10]:
import pandas as pd
import numpy as np
import torch
import ast
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. โหลด Dataset จริงจาก clean-dataset/
# ==========================================
df_shopee = pd.read_csv("clean-dataset/labeled_thai_clean_dataset_shopee.csv")
df_x      = pd.read_csv("clean-dataset/labeled_thai_clean_dataset_X.csv")
df = pd.concat([df_shopee, df_x], ignore_index=True).dropna(subset=['clean_text', 'emotion_tags'])

print(f"โหลดข้อมูลสำเร็จ: {len(df)} rows")

# ==========================================
# 2. กำหนด Label Classes (ลำดับต้องคงที่เสมอ)
# ==========================================
LABEL_CLASSES = [
    "Joy (ดีใจ)",
    "Sadness (เศร้า)",
    "Anger (โกรธ)",
    "Fear (กลัว)",
    "Surprise (ประหลาดใจ)",
    "Neutral (เฉยๆ)",
    "Disgust (ความขยะแขยง)"
]
NUM_LABELS = len(LABEL_CLASSES)  # 7
label2idx  = {lbl: i for i, lbl in enumerate(LABEL_CLASSES)}

print(f"จำนวน Labels: {NUM_LABELS}")
print(f"Labels: {LABEL_CLASSES}")

# ==========================================
# 3. แปลง emotion_tags → Binary Vector [1,0,0,0,1,0,0]
# ==========================================
def encode_labels(tag_str):
    """แปลง '["Joy (ดีใจ)", "Neutral (เฉยๆ)"]' → [1,0,0,0,0,1,0]"""
    try:
        tags = ast.literal_eval(tag_str)
    except:
        tags = []
    vec = [0.0] * NUM_LABELS
    for tag in tags:
        if tag in label2idx:
            vec[label2idx[tag]] = 1.0
    return vec

df['labels'] = df['emotion_tags'].apply(encode_labels)

# ตรวจสอบตัวอย่าง
print("\nตัวอย่างข้อมูลที่พร้อมเทรน:")
print(df[['clean_text', 'emotion_tags', 'labels']].head(3).to_string())

# ==========================================
# 4. คำนวณ pos_weight สำหรับ Imbalanced Data
#    pos_weight[i] = (N - pos_i) / pos_i
# ==========================================
label_matrix = np.array(df['labels'].tolist())  # shape (N, 7)
pos_counts   = label_matrix.sum(axis=0)          # จำนวนข้อมูลบวกแต่ละ class
neg_counts   = len(df) - pos_counts
pos_weight   = neg_counts / (pos_counts + 1e-6)  # ป้องกัน div-by-zero

print("\n=== Class Distribution + pos_weight ===")
for i, lbl in enumerate(LABEL_CLASSES):
    print(f"  {lbl:<25} count={int(pos_counts[i]):>4}  pos_weight={pos_weight[i]:.2f}")

# ==========================================
# 5. ตั้งค่า Device (GPU ถ้ามี)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nกำลังรันบน: {device}")

# ==========================================
# 6. โหลด Tokenizers และ Models (3 ตัว)
# ==========================================
model_names = {
    "wangchan"  : "airesearch/wangchanberta-base-att-spm-uncased",
    "mbert"     : "bert-base-multilingual-cased",
    "xlm_roberta": "xlm-roberta-base"
}

tokenizers = {}
models     = {}

print("\nกำลังโหลด Models... (อาจใช้เวลาสักครู่ในครั้งแรก)")
for key, path in model_names.items():
    # wangchan ใช้ SentencePiece → ต้องปิด fast tokenizer
    use_fast = False if key == "wangchan" else True
    tokenizers[key] = AutoTokenizer.from_pretrained(path, use_fast=use_fast)
    models[key] = AutoModelForSequenceClassification.from_pretrained(
        path,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification"  # ใช้ BCEWithLogitsLoss อัตโนมัติ
    ).to(device)
    print(f"  ✅ {key} โหลดสำเร็จ")

print("\nโหลด Models ทั้งหมดสำเร็จ พร้อมลุย!")

โหลดข้อมูลสำเร็จ: 2000 rows
จำนวน Labels: 7
Labels: ['Joy (ดีใจ)', 'Sadness (เศร้า)', 'Anger (โกรธ)', 'Fear (กลัว)', 'Surprise (ประหลาดใจ)', 'Neutral (เฉยๆ)', 'Disgust (ความขยะแขยง)']

ตัวอย่างข้อมูลที่พร้อมเทรน:
                       clean_text    emotion_tags                               labels
0                        ใช้ได้ดี  ["Joy (ดีใจ)"]  [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
1                          โอเคดี  ["Joy (ดีใจ)"]  [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
2  ง่ายดี รอแอพไทยทำได้แบบนี้บ้าง  ["Joy (ดีใจ)"]  [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

=== Class Distribution + pos_weight ===
  Joy (ดีใจ)                count=1714  pos_weight=0.17
  Sadness (เศร้า)           count= 202  pos_weight=8.90
  Anger (โกรธ)              count= 239  pos_weight=7.37
  Fear (กลัว)               count=  21  pos_weight=94.24
  Surprise (ประหลาดใจ)      count=  64  pos_weight=30.25
  Neutral (เฉยๆ)            count=  60  pos_weight=32.33
  Disgust (ความขยะแขยง)     count=  35  pos_weight=56.14

ก

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ wangchan โหลดสำเร็จ


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ mbert โหลดสำเร็จ


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ xlm_roberta โหลดสำเร็จ

โหลด Models ทั้งหมดสำเร็จ พร้อมลุย!


ขอโค้ดสเตปต่อไป: วิธีการแปลงข้อความ (Tokenization) และลูปการ Fine-tune โมเดลด้วย PyTorch DataLoader หน่อย
สเตปนี้คือการสร้าง "ตัวจ่ายข้อมูล" (DataLoader) เพื่อแปลงข้อความให้เป็นตัวเลข (Tokenization) และเขียนลูปฝึกสอน (Training Loop) สำหรับ Fine-tune โมเดลครับ

ในการทำจริง คุณต้องรันการ Fine-tune แบบนี้ให้กับโมเดลทั้ง 3 ตัว (WangchanBERTa, mBERT, XLM-RoBERTa) แยกกัน โค้ดด้านล่างนี้จะแสดงโครงสร้างที่สามารถนำไปวนลูปใช้กับโมเดลทุกตัวได้เลย

In [13]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ==========================================
# 1. Dataset Class สำหรับงาน Multi-label
# ==========================================
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text  = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids'     : encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels'        : torch.FloatTensor(label)  # ต้องเป็น Float สำหรับ BCE Loss
        }

# ==========================================
# 2. ฟังก์ชัน Fine-tune (รองรับ pos_weight)
# ==========================================
def train_model(model, train_loader, val_loader, optimizer, device, epochs=3, pw=None):
    """
    pw: pos_weight tensor (shape [NUM_LABELS]) สำหรับ Imbalanced data
        ถ้า model มี problem_type="multi_label_classification" จะใช้ loss ภายใน
        แต่เราจะ override ด้วย BCEWithLogitsLoss + pos_weight แทน
    """
    criterion = torch.nn.BCEWithLogitsLoss(
        pos_weight=pw.to(device) if pw is not None else None
    )

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        total_loss = 0
        bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in bar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            model.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits  # shape (batch, NUM_LABELS)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            bar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_train_loss = total_loss / len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['labels'].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                val_loss += criterion(outputs.logits, labels).item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# ==========================================
# 3. แบ่ง Train / Validation (80:20)
# ==========================================
all_texts  = df['clean_text'].tolist()
all_labels = df['labels'].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    all_texts, all_labels,
    test_size=0.2,
    random_state=42
)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")

# pos_weight tensor จาก Cell ก่อนหน้า
pw_tensor = torch.FloatTensor(pos_weight)

# ==========================================
# 4. เริ่มเทรน (เลือกโมเดลที่ต้องการ)
# ==========================================
# เปลี่ยน key เป็น "mbert" หรือ "xlm_roberta" เพื่อเทรนโมเดลอื่น
MODEL_KEY = "wangchan"
#MODEL_KEY = "mbert"
#MODEL_KEY = "xlm_roberta"




current_tokenizer = tokenizers[MODEL_KEY]
current_model     = models[MODEL_KEY]

train_dataset = EmotionDataset(train_texts, train_labels, current_tokenizer, max_len=128)
val_dataset   = EmotionDataset(val_texts,   val_labels,   current_tokenizer, max_len=128)

train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=16, shuffle=False)

optimizer = AdamW(current_model.parameters(), lr=2e-5)

print(f"\n--- เริ่ม Fine-tune '{MODEL_KEY}' ด้วยข้อมูลจริง {len(train_texts)} rows ---")
print(f"Batch size: 16 | Epochs: 3 | max_len: 128")
print(f"pos_weight: {[f'{w:.1f}' for w in pw_tensor.tolist()]}\n")

train_model(
    model=current_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=3,
    pw=pw_tensor
)

print("\n✅ เทรนเสร็จสมบูรณ์!")

Train: 1600 | Val: 400

--- เริ่ม Fine-tune 'wangchan' ด้วยข้อมูลจริง 1600 rows ---
Batch size: 16 | Epochs: 3 | max_len: 128
pos_weight: ['0.2', '8.9', '7.4', '94.2', '30.2', '32.3', '56.1']



Epoch 1/3 [Train]: 100%|██████████| 100/100 [00:36<00:00,  2.71it/s, loss=1.0365]


Epoch 1 | Train Loss: 1.0540 | Val Loss: 1.0437


Epoch 2/3 [Train]: 100%|██████████| 100/100 [00:34<00:00,  2.87it/s, loss=0.9810]


Epoch 2 | Train Loss: 0.9019 | Val Loss: 1.1171


Epoch 3/3 [Train]: 100%|██████████| 100/100 [00:34<00:00,  2.86it/s, loss=1.4016]


Epoch 3 | Train Loss: 0.8504 | Val Loss: 1.1085

✅ เทรนเสร็จสมบูรณ์!


สเตป 3: วิธีดึงผลลัพธ์ไปสอน Random Forest

In [14]:
import torch
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, hamming_loss
from sklearn.model_selection import train_test_split

# ==========================================
# 1. ฟังก์ชันดึง Probability จากโมเดล BERT
# ==========================================
def get_model_probabilities(model, dataloader, device):
    """รัน inference แล้วคืน probability matrix (N, NUM_LABELS)"""
    model.eval()
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # Logits → Probability ด้วย Sigmoid (Multi-label)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.extend(probs)

    return np.array(all_probs)  # shape (N, 7)


# ==========================================
# 2. เตรียม DataLoader สำหรับ Inference (ทั้งชุด)
# ==========================================
all_texts  = df['clean_text'].tolist()
all_labels = df['labels'].tolist()

# แบ่ง Train/Test (80:20) — ใช้ random_state เดียวกับตอนเทรน BERT
train_texts, test_texts, train_labels, test_labels = train_test_split(
    all_texts, all_labels, test_size=0.2, random_state=42
)

print(f"Train: {len(train_texts)} | Test: {len(test_texts)}")

# สร้าง DataLoader สำหรับ Train และ Test แยกกัน
def make_loader(texts, labels, tokenizer, batch_size=32):
    dataset = EmotionDataset(texts, labels, tokenizer, max_len=128)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False)


# ==========================================
# 3. ดึง Probability จากทั้ง 3 โมเดล
#    แล้ว Concat เป็น Feature Vector 21 มิติ
#    [prob_wangchan(7) | prob_mbert(7) | prob_xlm(7)]
# ==========================================
print("\nกำลังดึง Probability จากโมเดลทั้ง 3...")

train_features_list = []
test_features_list  = []

for key in ["wangchan", "mbert", "xlm_roberta"]:
    tok   = tokenizers[key]
    model = models[key]

    train_loader_inf = make_loader(train_texts, train_labels, tok)
    test_loader_inf  = make_loader(test_texts,  test_labels,  tok)

    train_probs = get_model_probabilities(model, train_loader_inf, device)
    test_probs  = get_model_probabilities(model, test_loader_inf,  device)

    train_features_list.append(train_probs)
    test_features_list.append(test_probs)
    print(f"  ✅ {key}: train {train_probs.shape} | test {test_probs.shape}")

# Concat feature จากทั้ง 3 โมเดลเป็น 1 matrix
X_train = np.concatenate(train_features_list, axis=1)  # (N_train, 21)
X_test  = np.concatenate(test_features_list,  axis=1)  # (N_test,  21)
y_train = np.array(train_labels)  # (N_train, 7)
y_test  = np.array(test_labels)   # (N_test,  7)

print(f"\nFeature shape → Train: {X_train.shape} | Test: {X_test.shape}")


# ==========================================
# 4. เทรน Random Forest (Multi-label)
# ==========================================
print("\nกำลังเทรน Random Forest...")

rf_base = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',  # รับมือ Imbalanced data
    random_state=42,
    n_jobs=-1                  # ใช้ CPU ทุก core
)
# MultiOutputClassifier ทำให้ Random Forest รองรับ Multi-label
rf_model = MultiOutputClassifier(rf_base)
rf_model.fit(X_train, y_train)

print("✅ เทรน Random Forest เสร็จแล้ว!")


# ==========================================
# 5. ประเมินผล
# ==========================================
y_pred = rf_model.predict(X_test)  # (N_test, 7) — binary 0/1

print("\n=== ผลการประเมิน Random Forest ===")
print(f"Hamming Loss (ต่ำ = ดี): {hamming_loss(y_test, y_pred):.4f}")
print()
print(classification_report(
    y_test, y_pred,
    target_names=LABEL_CLASSES,
    zero_division=0
))


# ==========================================
# 6. (Optional) ดูตัวอย่างการทำนาย
# ==========================================
print("\n=== ตัวอย่างการทำนาย (5 รายการแรก) ===")
for i in range(5):
    true_lbls = [LABEL_CLASSES[j] for j, v in enumerate(y_test[i]) if v == 1]
    pred_lbls = [LABEL_CLASSES[j] for j, v in enumerate(y_pred[i]) if v == 1]
    print(f"  ข้อความ : {test_texts[i][:50]}...")
    print(f"  จริง    : {true_lbls}")
    print(f"  ทำนาย   : {pred_lbls}")
    print()

Train: 1600 | Test: 400

กำลังดึง Probability จากโมเดลทั้ง 3...
  ✅ wangchan: train (1600, 7) | test (400, 7)
  ✅ mbert: train (1600, 7) | test (400, 7)
  ✅ xlm_roberta: train (1600, 7) | test (400, 7)

Feature shape → Train: (1600, 21) | Test: (400, 21)

กำลังเทรน Random Forest...
✅ เทรน Random Forest เสร็จแล้ว!

=== ผลการประเมิน Random Forest ===
Hamming Loss (ต่ำ = ดี): 0.0875

                       precision    recall  f1-score   support

           Joy (ดีใจ)       0.95      0.90      0.92       342
      Sadness (เศร้า)       0.31      0.59      0.41        37
         Anger (โกรธ)       0.37      0.69      0.48        42
          Fear (กลัว)       0.00      0.00      0.00         7
 Surprise (ประหลาดใจ)       0.67      0.19      0.30        21
       Neutral (เฉยๆ)       0.00      0.00      0.00        17
Disgust (ความขยะแขยง)       0.33      0.40      0.36         5

            micro avg       0.73      0.77      0.75       471
            macro avg       0.38      0.40     

In [15]:
import os
import joblib

# ==========================================
# บันทึกโมเดลทั้งหมดหลัง Fine-tune เสร็จ
# ==========================================
SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# บันทึก BERT Models ทั้ง 3 ตัว
for key in ["wangchan", "mbert", "xlm_roberta"]:
    model_path = os.path.join(SAVE_DIR, f"{key}_model")
    tokenizer_path = os.path.join(SAVE_DIR, f"{key}_tokenizer")

    models[key].save_pretrained(model_path)
    tokenizers[key].save_pretrained(tokenizer_path)
    print(f"✅ บันทึก {key} → {model_path}")

# บันทึก Random Forest Meta-Classifier
rf_path = os.path.join(SAVE_DIR, "random_forest_meta.joblib")
joblib.dump(rf_model, rf_path)
print(f"✅ บันทึก Random Forest → {rf_path}")

# บันทึก Label Classes (สำคัญ! ต้องใช้ตอน inference)
import json as _json
with open(os.path.join(SAVE_DIR, "label_classes.json"), "w", encoding="utf-8") as f:
    _json.dump(LABEL_CLASSES, f, ensure_ascii=False, indent=2)
print(f"✅ บันทึก LABEL_CLASSES → {SAVE_DIR}/label_classes.json")

print(f"\n🎉 บันทึกโมเดลทั้งหมดลงใน '{SAVE_DIR}/' เรียบร้อย!")
print(f"   ไฟล์ที่ได้:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(f"   📁 {f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ บันทึก wangchan → saved_models/wangchan_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ บันทึก mbert → saved_models/mbert_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ บันทึก xlm_roberta → saved_models/xlm_roberta_model
✅ บันทึก Random Forest → saved_models/random_forest_meta.joblib
✅ บันทึก LABEL_CLASSES → saved_models/label_classes.json

🎉 บันทึกโมเดลทั้งหมดลงใน 'saved_models/' เรียบร้อย!
   ไฟล์ที่ได้:
   📁 label_classes.json
   📁 mbert_model
   📁 mbert_tokenizer
   📁 random_forest_meta.joblib
   📁 wangchan_model
   📁 wangchan_tokenizer
   📁 xlm_roberta_model
   📁 xlm_roberta_tokenizer
